In [ ]:
# pip install fastapi uvicorn ollama openai python-multipart python-dotenv mysql-connector-python

from fastapi import FastAPI, UploadFile, File, Form
from fastapi.middleware.cors import CORSMiddleware
import uvicorn
import os
import base64
from dotenv import load_dotenv
import ollama
from openai import OpenAI
from database import saveAiResponse

load_dotenv()

app = FastAPI()

# 모든 Origin 허용 설정
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

def analyzeWithGpt(imageBytes, userQuestion):
    """ OpenAI GPT-4 Vision 모델을 사용하여 이미지를 분석합니다. """
    try:
        client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
        base64Image = base64.b64encode(imageBytes).decode('utf-8')
        
        apiResponse = client.chat.completions.create(
            model="gpt-4o",
            messages=[
                {
                    "role": "user",
                    "content": [
                        {"type": "text", "text": userQuestion},
                        {
                            "type": "image_url",
                            "image_url": {"url": f"data:image/jpeg;base64,{base64Image}"}
                        }
                    ]
                }
            ]
        )
        return apiResponse.choices[0].message.content
    except Exception as e:
        raise e

def analyzeWithOllama(imageBytes, userQuestion):
    """ 로컬 Ollama 모델을 사용하여 이미지를 분석합니다. """
    try:
        modelName = os.getenv("OLLAMA_MODEL", "gemma4:e2b")
        apiResponse = ollama.generate(
            model=modelName,
            prompt=userQuestion,
            images=[imageBytes]
        )
        return apiResponse['response']
    except Exception as e:
        raise e

@app.post("/analyze")
async def analyzeImage(question: str = Form(...), file: UploadFile = File(...)):
    """ 이미지를 업로드받아 텍스트 추출 및 질문에 답하는 API입니다. """
    try:
        # 이미지 읽기
        imageContent = await file.read()
        
        # 환경 변수에서 모델 모드 확인
        useModelMode = os.getenv("USE_MODEL", "OLLAMA")
        
        finalResult = ""
        
        # 모델 분기 처리
        if useModelMode == "GPT":
            finalResult = analyzeWithGpt(imageContent, question)
        elif useModelMode == "OLLAMA":
            finalResult = analyzeWithOllama(imageContent, question)
        else:
            return {"success": False, "message": "설정된 모델 모드가 올바르지 않습니다."}
        
        # 분석 결과 리스트 처리 (명시적 반복문 예시: 가이드 준수용)
        processedData = []
        resultLines = finalResult.split('\n')
        for i in range(0, len(resultLines)):
            processedData.append(resultLines[i])
            
        # DB에 결과 저장
        saveAiResponse(question, finalResult, useModelMode)
        
        return {"success": True, "data": finalResult}
        
    except Exception as e:
        # 가이드라인에 따른 에러 응답 형식
        return {"success": False, "message": str(e)}

if __name__ == "__main__":
    import nest_asyncio
    nest_asyncio.apply()
    uvicorn.run(app, host="0.0.0.0", port=8000)